# Project 4: Custom PyTorch CUDA Extension
### Swish + Mish + Fused Bias+Swish — hand-written CUDA backward passes

**Five files we build:**
- `swish_cuda.cu` — CUDA kernels (forward + backward for Swish, Mish, FusedBiasSwish)
- `swish_cuda.cpp` — pybind11 C++ binding layer
- `swish.py` — Python `autograd.Function` + `nn.Module` drop-ins
- `test_swish.py` — 10 correctness tests + 7 benchmarks
- `setup.py` — pip-installable build config

**Run order:** Cells 1 → 12 in order. Compile in Cell 4 takes ~2 min.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader
!nvcc --version | grep release
import torch
cap = torch.cuda.get_device_capability()
print(f'PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0)}  arch: sm_{cap[0]}{cap[1]}')

In [ ]:
import os
os.makedirs('swish_ext', exist_ok=True)
os.chdir('swish_ext')

with open('swish_cuda.cu', 'w') as f:
    f.write(r'''#include <cuda.h>
#include <cuda_runtime.h>
#include <torch/extension.h>

#define BLOCK_SIZE 256

template <typename T>
__device__ __forceinline__ T sigmoid_(T x) { return (T)1.0 / ((T)1.0 + exp(-x)); }

template <typename T>
__device__ __forceinline__ T swish_f(T x) { return x * sigmoid_(x); }

template <typename T>
__device__ __forceinline__ T swish_d(T x) {
    T s = sigmoid_(x), sw = x * s;
    return s + sw * ((T)1.0 - s);
}

template <typename T>
__device__ __forceinline__ T softplus_(T x) {
    return (x > (T)20.0) ? x : log((T)1.0 + exp(x));
}

template <typename T>
__device__ __forceinline__ T mish_f(T x) { return x * tanh(softplus_(x)); }

template <typename T>
__device__ __forceinline__ T mish_d(T x) {
    T sp = softplus_(x), t = tanh(sp);
    return t + x * ((T)1.0 - t * t) * sigmoid_(x);
}

/* Swish forward */
template<typename T>
__global__ void swish_fwd_k(const T* x, T* y, int64_t N) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) y[i] = swish_f(x[i]);
}

/* Swish backward */
template<typename T>
__global__ void swish_bwd_k(const T* x, const T* go, T* gi, int64_t N) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) gi[i] = go[i] * swish_d(x[i]);
}

/* Mish forward */
template<typename T>
__global__ void mish_fwd_k(const T* x, T* y, int64_t N) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) y[i] = mish_f(x[i]);
}

/* Mish backward */
template<typename T>
__global__ void mish_bwd_k(const T* x, const T* go, T* gi, int64_t N) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) gi[i] = go[i] * mish_d(x[i]);
}

/* Fused bias+swish forward: y = swish(x + bias[i % C]) */
template<typename T>
__global__ void fbs_fwd_k(const T* x, const T* b, T* y, int64_t N, int64_t C) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) { T v = x[i] + b[i % C]; y[i] = swish_f(v); }
}

/* Fused bias+swish backward: grad_x + grad_bias (atomicAdd) */
template<typename T>
__global__ void fbs_bwd_k(const T* x, const T* b, const T* go,
                           T* gx, T* gb, int64_t N, int64_t C) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        T v = x[i] + b[i % C];
        T g = go[i] * swish_d(v);
        gx[i] = g;
        atomicAdd(&gb[i % C], g);
    }
}

/* float4 vectorised swish */
__global__ void swish_vec4_k(const float* x, float* y, int64_t N) {
    int64_t i4 = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    int64_t i  = i4 * 4;
    if (i + 3 < N) {
        float4 vx = reinterpret_cast<const float4*>(x)[i4];
        float4 vy;
        vy.x = swish_f(vx.x); vy.y = swish_f(vx.y);
        vy.z = swish_f(vx.z); vy.w = swish_f(vx.w);
        reinterpret_cast<float4*>(y)[i4] = vy;
    } else {
        for (int64_t j = i; j < N && j < i + 4; ++j) y[j] = swish_f(x[j]);
    }
}

/* C++ launchers */
static int blk(int64_t N) { return (int)((N + BLOCK_SIZE - 1) / BLOCK_SIZE); }

torch::Tensor swish_forward_cuda(torch::Tensor x) {
    auto y = torch::empty_like(x); int64_t N = x.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES_AND_HALF(x.scalar_type(), "swish_fwd", ([&] {
        swish_fwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), y.data_ptr<scalar_t>(), N);
    }));
    return y;
}

torch::Tensor swish_backward_cuda(torch::Tensor x, torch::Tensor go) {
    auto gi = torch::empty_like(x); int64_t N = x.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES_AND_HALF(x.scalar_type(), "swish_bwd", ([&] {
        swish_bwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), go.data_ptr<scalar_t>(), gi.data_ptr<scalar_t>(), N);
    }));
    return gi;
}

torch::Tensor mish_forward_cuda(torch::Tensor x) {
    auto y = torch::empty_like(x); int64_t N = x.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES_AND_HALF(x.scalar_type(), "mish_fwd", ([&] {
        mish_fwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), y.data_ptr<scalar_t>(), N);
    }));
    return y;
}

torch::Tensor mish_backward_cuda(torch::Tensor x, torch::Tensor go) {
    auto gi = torch::empty_like(x); int64_t N = x.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES_AND_HALF(x.scalar_type(), "mish_bwd", ([&] {
        mish_bwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), go.data_ptr<scalar_t>(), gi.data_ptr<scalar_t>(), N);
    }));
    return gi;
}

torch::Tensor fbs_forward_cuda(torch::Tensor x, torch::Tensor b) {
    auto y = torch::empty_like(x); int64_t N = x.numel(), C = b.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "fbs_fwd", ([&] {
        fbs_fwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), b.data_ptr<scalar_t>(), y.data_ptr<scalar_t>(), N, C);
    }));
    return y;
}

std::vector<torch::Tensor> fbs_backward_cuda(torch::Tensor x, torch::Tensor b, torch::Tensor go) {
    auto gx = torch::empty_like(x), gb = torch::zeros_like(b);
    int64_t N = x.numel(), C = b.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "fbs_bwd", ([&] {
        fbs_bwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), b.data_ptr<scalar_t>(), go.data_ptr<scalar_t>(),
            gx.data_ptr<scalar_t>(), gb.data_ptr<scalar_t>(), N, C);
    }));
    return {gx, gb};
}

torch::Tensor swish_vec4_cuda(torch::Tensor x) {
    auto y = torch::empty_like(x); int64_t N = x.numel(), N4 = (N + 3) / 4;
    auto s = at::cuda::getCurrentCUDAStream();
    swish_vec4_k<<<blk(N4), BLOCK_SIZE, 0, s>>>(x.data_ptr<float>(), y.data_ptr<float>(), N);
    return y;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("swish_forward",  &swish_forward_cuda);
    m.def("swish_backward", &swish_backward_cuda);
    m.def("mish_forward",   &mish_forward_cuda);
    m.def("mish_backward",  &mish_backward_cuda);
    m.def("fbs_forward",    &fbs_forward_cuda);
    m.def("fbs_backward",   &fbs_backward_cuda);
    m.def("swish_vec4",     &swish_vec4_cuda);
}''')
print('Written swish_cuda.cu  (%d lines)' % len(r'''#include <cuda.h>
#include <cuda_runtime.h>
#include <torch/extension.h>

#define BLOCK_SIZE 256

template <typename T>
__device__ __forceinline__ T sigmoid_(T x) { return (T)1.0 / ((T)1.0 + exp(-x)); }

template <typename T>
__device__ __forceinline__ T swish_f(T x) { return x * sigmoid_(x); }

template <typename T>
__device__ __forceinline__ T swish_d(T x) {
    T s = sigmoid_(x), sw = x * s;
    return s + sw * ((T)1.0 - s);
}

template <typename T>
__device__ __forceinline__ T softplus_(T x) {
    return (x > (T)20.0) ? x : log((T)1.0 + exp(x));
}

template <typename T>
__device__ __forceinline__ T mish_f(T x) { return x * tanh(softplus_(x)); }

template <typename T>
__device__ __forceinline__ T mish_d(T x) {
    T sp = softplus_(x), t = tanh(sp);
    return t + x * ((T)1.0 - t * t) * sigmoid_(x);
}

/* Swish forward */
template<typename T>
__global__ void swish_fwd_k(const T* x, T* y, int64_t N) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) y[i] = swish_f(x[i]);
}

/* Swish backward */
template<typename T>
__global__ void swish_bwd_k(const T* x, const T* go, T* gi, int64_t N) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) gi[i] = go[i] * swish_d(x[i]);
}

/* Mish forward */
template<typename T>
__global__ void mish_fwd_k(const T* x, T* y, int64_t N) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) y[i] = mish_f(x[i]);
}

/* Mish backward */
template<typename T>
__global__ void mish_bwd_k(const T* x, const T* go, T* gi, int64_t N) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) gi[i] = go[i] * mish_d(x[i]);
}

/* Fused bias+swish forward: y = swish(x + bias[i % C]) */
template<typename T>
__global__ void fbs_fwd_k(const T* x, const T* b, T* y, int64_t N, int64_t C) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) { T v = x[i] + b[i % C]; y[i] = swish_f(v); }
}

/* Fused bias+swish backward: grad_x + grad_bias (atomicAdd) */
template<typename T>
__global__ void fbs_bwd_k(const T* x, const T* b, const T* go,
                           T* gx, T* gb, int64_t N, int64_t C) {
    int64_t i = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        T v = x[i] + b[i % C];
        T g = go[i] * swish_d(v);
        gx[i] = g;
        atomicAdd(&gb[i % C], g);
    }
}

/* float4 vectorised swish */
__global__ void swish_vec4_k(const float* x, float* y, int64_t N) {
    int64_t i4 = (int64_t)blockIdx.x * blockDim.x + threadIdx.x;
    int64_t i  = i4 * 4;
    if (i + 3 < N) {
        float4 vx = reinterpret_cast<const float4*>(x)[i4];
        float4 vy;
        vy.x = swish_f(vx.x); vy.y = swish_f(vx.y);
        vy.z = swish_f(vx.z); vy.w = swish_f(vx.w);
        reinterpret_cast<float4*>(y)[i4] = vy;
    } else {
        for (int64_t j = i; j < N && j < i + 4; ++j) y[j] = swish_f(x[j]);
    }
}

/* C++ launchers */
static int blk(int64_t N) { return (int)((N + BLOCK_SIZE - 1) / BLOCK_SIZE); }

torch::Tensor swish_forward_cuda(torch::Tensor x) {
    auto y = torch::empty_like(x); int64_t N = x.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES_AND_HALF(x.scalar_type(), "swish_fwd", ([&] {
        swish_fwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), y.data_ptr<scalar_t>(), N);
    }));
    return y;
}

torch::Tensor swish_backward_cuda(torch::Tensor x, torch::Tensor go) {
    auto gi = torch::empty_like(x); int64_t N = x.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES_AND_HALF(x.scalar_type(), "swish_bwd", ([&] {
        swish_bwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), go.data_ptr<scalar_t>(), gi.data_ptr<scalar_t>(), N);
    }));
    return gi;
}

torch::Tensor mish_forward_cuda(torch::Tensor x) {
    auto y = torch::empty_like(x); int64_t N = x.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES_AND_HALF(x.scalar_type(), "mish_fwd", ([&] {
        mish_fwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), y.data_ptr<scalar_t>(), N);
    }));
    return y;
}

torch::Tensor mish_backward_cuda(torch::Tensor x, torch::Tensor go) {
    auto gi = torch::empty_like(x); int64_t N = x.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES_AND_HALF(x.scalar_type(), "mish_bwd", ([&] {
        mish_bwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), go.data_ptr<scalar_t>(), gi.data_ptr<scalar_t>(), N);
    }));
    return gi;
}

torch::Tensor fbs_forward_cuda(torch::Tensor x, torch::Tensor b) {
    auto y = torch::empty_like(x); int64_t N = x.numel(), C = b.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "fbs_fwd", ([&] {
        fbs_fwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), b.data_ptr<scalar_t>(), y.data_ptr<scalar_t>(), N, C);
    }));
    return y;
}

std::vector<torch::Tensor> fbs_backward_cuda(torch::Tensor x, torch::Tensor b, torch::Tensor go) {
    auto gx = torch::empty_like(x), gb = torch::zeros_like(b);
    int64_t N = x.numel(), C = b.numel();
    auto s = at::cuda::getCurrentCUDAStream();
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "fbs_bwd", ([&] {
        fbs_bwd_k<scalar_t><<<blk(N), BLOCK_SIZE, 0, s>>>(
            x.data_ptr<scalar_t>(), b.data_ptr<scalar_t>(), go.data_ptr<scalar_t>(),
            gx.data_ptr<scalar_t>(), gb.data_ptr<scalar_t>(), N, C);
    }));
    return {gx, gb};
}

torch::Tensor swish_vec4_cuda(torch::Tensor x) {
    auto y = torch::empty_like(x); int64_t N = x.numel(), N4 = (N + 3) / 4;
    auto s = at::cuda::getCurrentCUDAStream();
    swish_vec4_k<<<blk(N4), BLOCK_SIZE, 0, s>>>(x.data_ptr<float>(), y.data_ptr<float>(), N);
    return y;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("swish_forward",  &swish_forward_cuda);
    m.def("swish_backward", &swish_backward_cuda);
    m.def("mish_forward",   &mish_forward_cuda);
    m.def("mish_backward",  &mish_backward_cuda);
    m.def("fbs_forward",    &fbs_forward_cuda);
    m.def("fbs_backward",   &fbs_backward_cuda);
    m.def("swish_vec4",     &swish_vec4_cuda);
}'''.splitlines()))

In [ ]:
import torch
cap = torch.cuda.get_device_capability()
arch = f'sm_{cap[0]}{cap[1]}'

setup_src = f'''
from setuptools import setup
from torch.utils.cpp_extension import BuildExtension, CUDAExtension
setup(
    name='swish_cuda',
    ext_modules=[CUDAExtension(
        name='swish_cuda',
        sources=['swish_cuda.cu'],
        extra_compile_args={{
            'cxx': ['-O2', '-std=c++17'],
            'nvcc': ['-O2', f'-arch={arch}', '--use_fast_math',
                     '--maxrregcount=64', '-std=c++17']
        }}
    )],
    cmdclass={'build_ext': BuildExtension}
)
'''

with open('setup.py', 'w') as f:
    f.write(setup_src)
print(f'Written setup.py  (arch={arch})')

In [ ]:
# This takes ~2-3 minutes — nvcc compiling CUDA C++
!pip install -e . --quiet 2>&1 | tail -5
print('Build complete!')

In [ ]:
import importlib, sys
# Force fresh import if already loaded
if 'swish_cuda' in sys.modules:
    del sys.modules['swish_cuda']

import swish_cuda, torch, torch.nn.functional as F
print('Extension imported:', dir(swish_cuda))

x = torch.randn(50_000, device='cuda')
err = (swish_cuda.swish_forward(x) - F.silu(x)).abs().max().item()
print(f'Swish max error vs F.silu: {err:.2e}  (expect < 1e-5)')

err_m = (swish_cuda.mish_forward(x) - x * torch.tanh(F.softplus(x))).abs().max().item()
print(f'Mish  max error vs PyTorch: {err_m:.2e}  (expect < 1e-4)')

In [ ]:
swish_src = '''import math, warnings\nimport torch, torch.nn as nn, torch.nn.functional as F\nfrom torch import Tensor\n\ntry:\n    import swish_cuda as _C\n    _EXT = True\nexcept ImportError:\n    _C = None; _EXT = False\n    warnings.warn("swish_cuda not built — using PyTorch fallback")\n\n# Pure-PyTorch helpers (used for fallback + float64 gradcheck)\ndef _swish_pt(x):    return x * torch.sigmoid(x)\ndef _mish_pt(x):     return x * torch.tanh(F.softplus(x))\ndef _swish_d_pt(x):\n    s = torch.sigmoid(x); sw = x * s; return s + sw * (1.0 - s)\ndef _mish_d_pt(x):\n    sp = F.softplus(x); t = torch.tanh(sp)\n    return t + x * (1.0 - t**2) * torch.sigmoid(x)\n\n# Expose for test file\n_swish_pytorch = _swish_pt\n_mish_pytorch  = _mish_pt\n\nclass SwishFunction(torch.autograd.Function):\n    """f(x) = x * sigmoid(x).  Backward computed from x alone."""\n    @staticmethod\n    def forward(ctx, x):\n        ctx.save_for_backward(x)\n        if _EXT and x.is_cuda and x.dtype in (torch.float32, torch.float16):\n            return _C.swish_forward(x.contiguous())\n        return _swish_pt(x)\n    @staticmethod\n    def backward(ctx, go):\n        x, = ctx.saved_tensors\n        if _EXT and x.is_cuda and x.dtype == torch.float32:\n            return _C.swish_backward(x.contiguous(), go.contiguous())\n        return go * _swish_d_pt(x)\n\nclass MishFunction(torch.autograd.Function):\n    """f(x) = x * tanh(softplus(x)).  Backward computed from x alone."""\n    @staticmethod\n    def forward(ctx, x):\n        ctx.save_for_backward(x)\n        if _EXT and x.is_cuda and x.dtype in (torch.float32, torch.float16):\n            return _C.mish_forward(x.contiguous())\n        return _mish_pt(x)\n    @staticmethod\n    def backward(ctx, go):\n        x, = ctx.saved_tensors\n        if _EXT and x.is_cuda and x.dtype == torch.float32:\n            return _C.mish_backward(x.contiguous(), go.contiguous())\n        return go * _mish_d_pt(x)\n\nclass FusedBiasSwishFunction(torch.autograd.Function):\n    """f(x, bias) = swish(x + bias).  Bias add fused into activation."""\n    @staticmethod\n    def forward(ctx, x, bias):\n        ctx.save_for_backward(x, bias)\n        if _EXT and x.is_cuda and x.dtype == torch.float32:\n            return _C.fbs_forward(x.contiguous(), bias.contiguous())\n        return _swish_pt(x + bias)\n    @staticmethod\n    def backward(ctx, go):\n        x, bias = ctx.saved_tensors\n        if _EXT and x.is_cuda and x.dtype == torch.float32:\n            gx, gb = _C.fbs_backward(x.contiguous(), bias.contiguous(), go.contiguous())\n            return gx, gb\n        z = x + bias; da = _swish_d_pt(z)\n        return go * da, (go * da).reshape(-1, bias.shape[0]).sum(0)\n\n# Functional API\ndef swish(x):              return SwishFunction.apply(x)\ndef mish(x):               return MishFunction.apply(x)\ndef fused_bias_swish(x,b): return FusedBiasSwishFunction.apply(x, b)\n\n# nn.Module API\nclass Swish(nn.Module):\n    def forward(self, x): return swish(x)\n    def extra_repr(self): return f"backend={\'cuda_ext\' if _EXT else \'pytorch\'}"\n\nclass Mish(nn.Module):\n    def forward(self, x): return mish(x)\n    def extra_repr(self): return f"backend={\'cuda_ext\' if _EXT else \'pytorch\'}"\n\nclass FusedBiasSwish(nn.Module):\n    def __init__(self, features: int):\n        super().__init__()\n        self.features = features\n        self.bias = nn.Parameter(torch.zeros(features))\n        nn.init.uniform_(self.bias, -1/features**0.5, 1/features**0.5)\n    def forward(self, x): return fused_bias_swish(x, self.bias)\n    def extra_repr(self): return f"features={self.features}"'''
with open('swish.py', 'w') as f:
    f.write(swish_src)
print(f'Written swish.py ({len(swish_src.splitlines())} lines)')

In [ ]:
import importlib, sys, torch, torch.nn.functional as F
for m in ['swish_cuda','swish']:
    if m in sys.modules: del sys.modules[m]

from swish import swish, mish, fused_bias_swish, _swish_pytorch, _mish_pytorch

def chk(name, a, b, atol=1e-4):
    err = (a.float()-b.float()).abs().max().item()
    ok  = err < atol
    print(f"  [{'PASS' if ok else 'FAIL'}] {name:<40}  err={err:.2e}")
    return ok

print('\n── Correctness ──────────────────────────────────────────────')
x = torch.randn(100_000, device='cuda')
chk('Swish float32 vs F.silu',     swish(x),        F.silu(x))
chk('Swish float32 vs _swish_pt',  swish(x),        _swish_pytorch(x))
chk('Swish float16',               swish(x.half()).float(), _swish_pytorch(x.half()).float(), atol=5e-3)
chk('Mish  float32',               mish(x),         _mish_pytorch(x))

B, C = 64, 512
xb, bi = torch.randn(B, C, device='cuda'), torch.randn(C, device='cuda')
chk('FusedBiasSwish [64,512]',      fused_bias_swish(xb,bi), _swish_pytorch(xb+bi))

print('\n── Boundary values ──────────────────────────────────────────')
xbnd = torch.tensor([-100.,-10.,-1.,0.,1.,10.,100.], device='cuda')
chk('Swish boundary',  swish(xbnd),  _swish_pytorch(xbnd))
chk('Mish boundary',   mish(xbnd),   _mish_pytorch(xbnd))

print('\n── Shape invariance ─────────────────────────────────────────')
for shape in [(100,),(16,64),(4,8,32),(2,4,8,16)]:
    xs = torch.randn(*shape, device='cuda')
    chk(f'Swish shape={list(shape)}', swish(xs), _swish_pytorch(xs))

In [ ]:
import torch
from swish import SwishFunction, MishFunction, FusedBiasSwishFunction

print('\n── Gradient Checks (torch.autograd.gradcheck) ───────────────')
print('  Uses float64 + numerical finite differences to verify backward formulas')

dev = 'cuda'

x = torch.randn(20, dtype=torch.float64, device=dev, requires_grad=True)
ok = torch.autograd.gradcheck(SwishFunction.apply, (x,), eps=1e-6, atol=1e-4)
print(f"  [{'PASS' if ok else 'FAIL'}] Swish gradcheck")

x = torch.randn(20, dtype=torch.float64, device=dev, requires_grad=True)
ok = torch.autograd.gradcheck(MishFunction.apply, (x,), eps=1e-6, atol=1e-4)
print(f"  [{'PASS' if ok else 'FAIL'}] Mish gradcheck")

x  = torch.randn(8, 16, dtype=torch.float64, device=dev, requires_grad=True)
bi = torch.randn(16,    dtype=torch.float64, device=dev, requires_grad=True)
ok = torch.autograd.gradcheck(FusedBiasSwishFunction.apply, (x, bi), eps=1e-6, atol=1e-4)
print(f"  [{'PASS' if ok else 'FAIL'}] FusedBiasSwish gradcheck (grad_x + grad_bias)")

print('\n  gradcheck passes = our backward math is analytically correct')

In [ ]:
import torch, torch.nn.functional as F
from swish import swish, _swish_pytorch

print('\n── Backward values vs PyTorch autograd ──────────────────────')
x_ref  = torch.randn(50_000, device='cuda', requires_grad=True)
x_ours = x_ref.clone().detach().requires_grad_(True)
go     = torch.ones(50_000, device='cuda')

_swish_pytorch(x_ref).backward(go)
swish(x_ours).backward(go)

err = (x_ref.grad - x_ours.grad).abs().max().item()
ok  = err < 1e-4
print(f"  [{'PASS' if ok else 'FAIL'}] Swish gradient values  max_err={err:.2e}")

In [ ]:
import torch, torch.nn as nn
from swish import Swish, FusedBiasSwish

print('\n── Training loop: Swish MLP vs nn.SiLU ─────────────────────')
torch.manual_seed(42)
device, B, D = 'cuda', 32, 256

def make_mlp(act_cls):
    return nn.Sequential(
        nn.Linear(D,D), act_cls(), nn.Linear(D,D), act_cls(), nn.Linear(D,10)
    ).to(device)

m_ours = make_mlp(Swish)
torch.manual_seed(42)
m_ref  = make_mlp(nn.SiLU)
m_ref.load_state_dict(m_ours.state_dict())

opt_o = torch.optim.Adam(m_ours.parameters(), lr=1e-3)
opt_r = torch.optim.Adam(m_ref.parameters(),  lr=1e-3)
crit  = nn.CrossEntropyLoss()

lo_list, lr_list = [], []
torch.manual_seed(0)
for _ in range(10):
    x = torch.randn(B, D, device=device)
    y = torch.randint(0, 10, (B,), device=device)
    for m, o, ls in [(m_ours,opt_o,lo_list),(m_ref,opt_r,lr_list)]:
        o.zero_grad(); l=crit(m(x),y); l.backward(); o.step(); ls.append(round(l.item(),4))

diff = abs(lo_list[-1] - lr_list[-1])
print(f"  [{'PASS' if diff < 0.05 else 'FAIL'}] Swish matches nn.SiLU  loss_diff={diff:.4f}")
print(f'  Swish loss:  {lo_list[0]} → {lo_list[-1]}')
print(f'  SiLU  loss:  {lr_list[0]} → {lr_list[-1]}')

print('\n── Training loop: FusedBiasSwish ────────────────────────────')
class FusedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc  = nn.Linear(D, D, bias=False)
        self.act = FusedBiasSwish(D)
    def forward(self, x): return self.act(self.fc(x))

fm  = FusedModel().to(device)
opt = torch.optim.SGD(fm.parameters(), lr=0.01)
torch.manual_seed(0)
for _ in range(5):
    x = torch.randn(B, D, device=device)
    l = (fm(x) - torch.randn(B, D, device=device)).pow(2).mean()
    opt.zero_grad(); l.backward(); opt.step()

has_wg = fm.fc.weight.grad is not None
has_bg = fm.act.bias.grad is not None
print(f"  [{'PASS' if (has_wg and has_bg) else 'FAIL'}] Gradients flow to weight ({has_wg}) AND bias ({has_bg})")

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, time
import swish_cuda as _C
from swish import swish, mish, fused_bias_swish

def gpu_time(fn, warmup=3, reps=100):
    for _ in range(warmup): fn()
    torch.cuda.synchronize()
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(reps): fn()
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / reps

silu = nn.SiLU()
sw_res, fu_res = [], []

print('\n── Benchmark 1: Swish forward vs nn.SiLU ──────────────────────')
print(f'{"N":>10}  {"SiLU (ms)":>12}  {"Ours (ms)":>12}  {"Speedup":>9}  {"GELT/s":>8}')
print('─' * 58)
for N in [1_000, 100_000, 1_000_000, 10_000_000]:
    x = torch.randn(N, device='cuda')
    ts = gpu_time(lambda: silu(x))
    to = gpu_time(lambda: swish(x))
    su = ts/to; ge = N/(to*1e-3)/1e9
    print(f'{N:>10,}  {ts:>12.4f}  {to:>12.4f}  {su:>9.3f}x  {ge:>8.2f}')
    sw_res.append((N,ts,to,su))

print('\n── Benchmark 2: FusedBiasSwish vs unfused ──────────────────────')
print(f'{"Shape":>14}  {"Unfused (ms)":>14}  {"Fused (ms)":>12}  {"Speedup":>9}  {"Mem MB":>8}')
print('─' * 63)
for B,C in [(64,256),(64,1024),(256,1024),(64,4096)]:
    x  = torch.randn(B, C, device='cuda')
    bi = torch.randn(C,    device='cuda')
    tu = gpu_time(lambda: silu(x+bi))
    tf = gpu_time(lambda: fused_bias_swish(x,bi))
    su = tu/tf; mem = B*C*4/1e6
    print(f'[{B},{C}]{" ":>5}  {tu:>14.4f}  {tf:>12.4f}  {su:>9.3f}x  {mem:>6.2f}MB')
    fu_res.append((f'{B}x{C}',tu,tf,su,mem))

print('\n── Benchmark 3: Backward latency ────────────────────────────────')
print(f'{"N":>10}  {"SiLU bwd":>12}  {"Swish bwd":>12}  {"Speedup":>9}')
print('─' * 48)
for N in [100_000, 1_000_000, 10_000_000]:
    xs = torch.randn(N, device='cuda', requires_grad=True)
    xo = torch.randn(N, device='cuda', requires_grad=True)
    go = torch.ones(N, device='cuda')
    ts = gpu_time(lambda: silu(xs).backward(go, retain_graph=True), reps=50)
    to = gpu_time(lambda: swish(xo).backward(go, retain_graph=True), reps=50)
    print(f'{N:>10,}  {ts:>12.4f}  {to:>12.4f}  {ts/to:>9.3f}x')

print('\n── Benchmark 4: float32 vs float16 ──────────────────────────────')
for dt, name in [(torch.float32,'float32'),(torch.float16,'float16')]:
    x = torch.randn(10_000_000, device='cuda', dtype=dt)
    t = gpu_time(lambda: swish(x))
    print(f'  {name}: {t:.3f} ms  ({10_000_000/(t*1e-3)/1e9:.2f} GELT/s)')

print('\n── Benchmark 5: float4 vectorised vs standard ───────────────────')
for N in [1_000_000, 10_000_000]:
    x = torch.randn(N, device='cuda', dtype=torch.float32)
    ts = gpu_time(lambda: _C.swish_forward(x))
    tv = gpu_time(lambda: _C.swish_vec4(x))
    print(f'  N={N//1_000_000}M: std={ts:.3f}ms  vec4={tv:.3f}ms  speedup={ts/tv:.2f}x')

# store for chart
globals()['_sw_res'] = sw_res
globals()['_fu_res'] = fu_res

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sw_res = globals()['_sw_res']
fu_res = globals()['_fu_res']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('white')

# Plot 1 — Swish latency
ax = axes[0]
Ns = [r[0] for r in sw_res]
labs = [f'{n//1000}K' if n<1e6 else f'{n//1_000_000}M' for n in Ns]
x = np.arange(len(Ns)); w = 0.35
ax.bar(x-w/2, [r[1] for r in sw_res], w, label='nn.SiLU',   color='#888888', alpha=0.85)
ax.bar(x+w/2, [r[2] for r in sw_res], w, label='Our Swish', color='#2a7abf', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labs, fontsize=9)
ax.set_yscale('log'); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_title('Swish forward vs nn.SiLU', fontweight='bold')
ax.set_ylabel('Latency ms (log scale)')

# Plot 2 — Fused latency
ax2 = axes[1]
shapes = [r[0] for r in fu_res]
x2 = np.arange(len(shapes))
ax2.bar(x2-w/2, [r[1] for r in fu_res], w, label='Unfused add+SiLU', color='#e07b39', alpha=0.85)
ax2.bar(x2+w/2, [r[2] for r in fu_res], w, label='Fused our kernel', color='#1a9e5c', alpha=0.85)
ax2.set_xticks(x2); ax2.set_xticklabels(shapes, fontsize=9)
ax2.legend(); ax2.grid(axis='y', alpha=0.3)
ax2.set_title('FusedBiasSwish vs unfused', fontweight='bold')
ax2.set_ylabel('Latency ms')

# Plot 3 — Speedup
ax3 = axes[2]
speedups = [r[3] for r in fu_res]
mems     = [r[4] for r in fu_res]
cols = ['#1a9e5c' if s>1 else '#e07b39' for s in speedups]
bars = ax3.bar(x2, speedups, 0.5, color=cols, alpha=0.85)
ax3.axhline(1, color='#888', linestyle='--', linewidth=1.2)
for bar, su, m in zip(bars, speedups, mems):
    ax3.text(bar.get_x()+bar.get_width()/2, su+0.02,
             f'{su:.2f}x\n-{m:.1f}MB', ha='center', fontsize=9, fontweight='bold')
ax3.set_xticks(x2); ax3.set_xticklabels(shapes, fontsize=9)
ax3.set_title('Fusion speedup (saved HBM in label)', fontweight='bold')
ax3.set_ylabel('Speedup over unfused')
ax3.grid(axis='y', alpha=0.3)

plt.suptitle('Custom CUDA Activation Extension — Benchmarks', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('swish_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved swish_benchmark.png')